In [1]:
"""
Genera df_subset_3000.csv:
  - 3000 imágenes AP/PA sampleadas (random, semilla fija)
  - full_path conservado tal cual viene del CSV original
  - Schema: dicom_id, subject_id, study_id + 14 labels + full_path + ViewPosition + calidad-imagen
"""

from pathlib import Path
import pandas as pd

# ---- Config ----------------------------------------------------------------
INPUT_PATH = Path(
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset-completo-merge\df_completo_view.csv"
)
OUTPUT_PATH = INPUT_PATH.with_name("df_subset_3000.csv")

SAMPLE_N = 3000
RANDOM_SEED = 42

LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax", "Support Devices",
]
# ---------------------------------------------------------------------------

def main() -> None:
    print(f"Leyendo: {INPUT_PATH}")
    df = pd.read_csv(INPUT_PATH)
    print(f"  shape: {df.shape}")

    # --- Solo AP/PA ----------------------------------------------------------
    is_ap_pa = df["ViewPosition"].isin(["AP", "PA"])
    print(f"\nAP/PA disponibles: {is_ap_pa.sum()}")

    # --- Sample --------------------------------------------------------------
    sample = df[is_ap_pa].sample(n=SAMPLE_N, random_state=RANDOM_SEED)
    print(f"Muestreado a:      {len(sample)} filas")

    # --- Schema de salida (full_path original, sin reconstruir) --------------
    output_cols = (
        ["dicom_id", "subject_id", "study_id"]
        + LABEL_COLS
        + ["full_path", "ViewPosition", "calidad-imagen"]
    )
    sample = sample[output_cols]

    sample.to_csv(OUTPUT_PATH, index=False)
    print(f"\nGuardado: {OUTPUT_PATH}")
    print(f"  shape: {sample.shape}")
    print(f"\nViewPosition distribución:")
    print(sample["ViewPosition"].value_counts().to_string())

if __name__ == "__main__":
    main()

Leyendo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_completo_view.csv
  shape: (377110, 25)

AP/PA disponibles: 243334
Muestreado a:      3000 filas

Guardado: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_subset_3000.csv
  shape: (3000, 20)

ViewPosition distribución:
ViewPosition
AP    1799
PA    1201


In [2]:
"""
Exploración de df_subset_3000.csv
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ---- Config ----------------------------------------------------------------
INPUT_PATH = Path(
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset-completo-merge\df_subset_3000.csv"
)

LABEL_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema",
    "Enlarged Cardiomediastinum", "Fracture", "Lung Lesion", "Lung Opacity",
    "No Finding", "Pleural Effusion", "Pleural Other", "Pneumonia",
    "Pneumothorax", "Support Devices",
]
# ---------------------------------------------------------------------------

df = pd.read_csv(INPUT_PATH)
print(f"Shape: {df.shape}")
print(f"Columnas: {df.columns.tolist()}\n")

# --- 1. ViewPosition --------------------------------------------------------
print("=" * 55)
print("1. DISTRIBUCIÓN ViewPosition")
print("=" * 55)
print(df["ViewPosition"].value_counts(dropna=False).to_string())
print()

# --- 2. Calidad-imagen ------------------------------------------------------
print("=" * 55)
print("2. CALIDAD-IMAGEN  (0=mala, 1=buena, NaN=no evaluada)")
print("=" * 55)
print(df["calidad-imagen"].value_counts(dropna=False).to_string())
print()

# --- 3. Hallazgos positivos por label ---------------------------------------
print("=" * 55)
print("3. HALLAZGOS POSITIVOS POR LABEL")
print("=" * 55)

rows = []
for col in LABEL_COLS:
    n_pos  = (df[col] == 1.0).sum()
    n_zero = (df[col] == 0.0).sum()
    n_neg1 = (df[col] == -1.0).sum()
    n_nan  = df[col].isna().sum()
    rows.append({
        "Label":    col,
        "1.0 (pos)":  n_pos,
        "% pos":    round(n_pos / len(df) * 100, 1),
        "0.0 (neg)":  n_zero,
        "-1.0 (inc)": n_neg1,
        "NaN":      n_nan,
    })

label_df = pd.DataFrame(rows).sort_values("1.0 (pos)", ascending=False)
print(label_df.to_string(index=False))
print()

# --- 4. Pacientes únicos (data leakage check) --------------------------------
print("=" * 55)
print("4. UNICIDAD")
print("=" * 55)
print(f"  Imágenes (filas)       : {len(df)}")
print(f"  dicom_id únicos        : {df['dicom_id'].nunique()}")
print(f"  subject_id únicos      : {df['subject_id'].nunique()}")
print(f"  study_id únicos        : {df['study_id'].nunique()}")
print()

# --- 5. Imágenes con calidad evaluada vs sin evaluar ------------------------
print("=" * 55)
print("5. COBERTURA DE CALIDAD")
print("=" * 55)
print(f"  Evaluadas (no NaN) : {df['calidad-imagen'].notna().sum()}")
print(f"  Sin evaluar (NaN)  : {df['calidad-imagen'].isna().sum()}")

Shape: (3000, 20)
Columnas: ['dicom_id', 'subject_id', 'study_id', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion', 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices', 'full_path', 'ViewPosition', 'calidad-imagen']

1. DISTRIBUCIÓN ViewPosition
ViewPosition
AP    1799
PA    1201

2. CALIDAD-IMAGEN  (0=mala, 1=buena, NaN=no evaluada)
calidad-imagen
1.0    2103
NaN     896
0.0       1

3. HALLAZGOS POSITIVOS POR LABEL
                     Label  1.0 (pos)  % pos  0.0 (neg)  -1.0 (inc)  NaN
                No Finding        993   33.1          0           0 2007
           Support Devices        902   30.1         53           3 2042
          Pleural Effusion        704   23.5        339          85 1872
              Lung Opacity        664   22.1         33          58 2245
               Atelectasis        602   20.1         20         143 2235
              

archivo de texto para descarga

In [3]:
"""
Genera subset3000_descarga.txt con rutas PhysioNet a partir de df_subset_3000.csv,
listo para usar con wget -i.

Formato por línea:
    files/pXX/p<subject_id>/s<study_id>/<dicom_id>.jpg
"""

from pathlib import Path
import pandas as pd

# ---- Config ----------------------------------------------------------------
INPUT_CSV = Path(
    r"C:\Users\trodr\Documents\proyecto-torax-v2.0"
    r"\01-dataset\dataset-completo-merge\df_subset_3000.csv"
)
OUTPUT_TXT = INPUT_CSV.with_name("subset3000_descarga.txt")
# ---------------------------------------------------------------------------


def build_physionet_path(row) -> str:
    subject_id = str(int(row["subject_id"]))
    study_id   = str(int(row["study_id"]))
    prefix     = subject_id[:2]
    return f"files/p{prefix}/p{subject_id}/s{study_id}/{row['dicom_id']}.jpg"


def main() -> None:
    print(f"Leyendo: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV)
    print(f"  shape: {df.shape}")

    paths = df.apply(build_physionet_path, axis=1).tolist()

    with open(OUTPUT_TXT, "w", newline="") as f:
        for p in paths:
            f.write(p + "\r\n")

    print(f"\nGuardado: {OUTPUT_TXT}")
    print(f"  líneas: {len(paths)}")
    print(f"\nPrimeras 3 líneas:")
    for p in paths[:3]:
        print(f"  {p}")


if __name__ == "__main__":
    main()

Leyendo: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\df_subset_3000.csv
  shape: (3000, 20)

Guardado: C:\Users\trodr\Documents\proyecto-torax-v2.0\01-dataset\dataset-completo-merge\subset3000_descarga.txt
  líneas: 3000

Primeras 3 líneas:
  files/p16/p16700191/s51325054/51adabdf-4234d20c-a63f7056-421c65ab-6dabfa41.jpg
  files/p18/p18452091/s58217747/89177690-364bac2f-5d496304-387f233c-db26c1fd.jpg
  files/p13/p13742148/s58404524/3b14f064-390ac420-34a6ad8c-351114dd-66126ab3.jpg
